# ShiftBench on Colab

Runtime → Change runtime type → **T4 GPU**. Run top to bottom.

1. **Setup**
2. **Sanity check** on the toy world (1 min, no downloads)
3. **Pilot A: ArmnetBench**, a public SO-101 dataset with success/failure labels. The "shift" here is *which policy is driving*: train the world model on human teleop demos, calibrate on them, then see whether the false-alarm promise holds on ACT, Diffusion, SmolVLA, π0, ... rollouts. You can run this **before the arm arrives**.
4. **Your own sessions** (after recording with `scripts/record_session.sh`)

In [ ]:
# 1. Setup. Push the repo to GitHub first, or upload the zip and unzip it here.
!git clone https://github.com/kishoren18-hue/shiftbench.git 2>/dev/null || echo "already cloned / upload the zip"
%cd shiftbench
!pip -q install -e . "lerobot[dataset]" "transformers>=4.53"
import torch; print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE - switch runtime to T4")

In [ ]:
# 2. Sanity check: the whole pipeline on synthetic latents (NOT robot results)
!python scripts/synthetic_experiment.py --out results/synthetic --k 30 --epochs 30

In [ ]:
# How fast is V-JEPA 2 here? (decides the live watchdog rate)
import time, numpy as np
from shiftbench.encoders import VJEPA2Encoder
enc = VJEPA2Encoder()            # downloads facebook/vjepa2-vitl-fpc64-256 (~1.3 GB) once
x = (np.random.rand(32, 480, 640, 3) * 255).astype(np.uint8)
enc.encode(x[:4]); t = time.time(); z = enc.encode(x); dt = time.time() - t
print(z.shape, f"{32/dt:.1f} frames/s")

## 3. Pilot A: ArmnetBench, `block_stack`, front camera

Downloads only the selected episodes. ~40 per policy keeps it under an hour on a T4.
The first cell prints the metadata columns and the (success, fail) counts per policy, so check them before the long encode.

In [ ]:
TASK = "Stack the colouful blocks"   # spelling as in the dataset
!python scripts/encode_dataset.py --repo_id armnet/armnetbench_v01_lerobot_so101 \
    --camera observation.images.front --cache cache/armnet_block_stack \
    --task "{TASK}" --condition_column policy_type --max_per_condition 40

If the counts above show no `teleoperated` episodes, encode the 50 reference demos from the task's teleop dataset instead and train on those:

In [ ]:
# only if needed:
# !python scripts/encode_dataset.py --repo_id villekuosmanen/armnetbench_block_stack \
#     --camera observation.images.front --cache cache/armnet_block_stack --all_success teleoperated

In [ ]:
!python scripts/train_detector.py --cache cache/armnet_block_stack --out runs/armnet_block_stack \
    --train_condition teleoperated --alpha 0.1 --train_frac 0.5
!python scripts/run_benchmark.py --cache cache/armnet_block_stack --detector runs/armnet_block_stack \
    --out results/armnet_block_stack --k 10

Read the output like this:
* **fixed** row far above the target on some policies = the published-style promise breaks when the policy changes.
* **adaptive_log** near the target everywhere = the fix works on real data.
* **AUROC** near 0.5 = the world model can't tell failures apart on this task/camera; try `--pool grid2`, the `top` camera, or a longer `--window`.

Caveat: policy rollouts start from the same scenes as the demos, so this is *policy* shift, not camera/lighting shift. It's a pilot, not the paper.

## 4. Your sessions

Copy your `data/` folder (all `rollout_shiftbench_*` sessions) and `data/sessions.txt` to Drive, then:

In [ ]:
from google.colab import drive; drive.mount('/content/drive')
!cp -r /content/drive/MyDrive/shiftbench_data/data .
!bash scripts/encode_sessions.sh cache/front observation.images.front
!python scripts/train_detector.py --cache cache/front --out runs/detector --alpha 0.05
!python scripts/run_benchmark.py --cache cache/front --detector runs/detector --out results/real --k 20 --drift_condition light_dim_1
!cp -r runs results /content/drive/MyDrive/shiftbench_data/